# Description

This script reads data, extracts mails of people using HunterIO and Dropcontact and writes it back to google sheet

In [20]:
!sudo /bin/bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

In [3]:
import logging

import ck_marketing.hunterio.hunter_api as cmhuhuap
import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hprint as hprint
from ck_marketing.hunterio.hunter_api import GoogleSheetsHelper

In [4]:
# Configure logger.
hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Print system signature.
_LOG.info("%s", henv.get_system_signature()[0])

# Configure the notebook style.
hprint.config_notebook()

DEBUG:helpers.hsystem:> (cd . && cd "$(git rev-parse --show-toplevel)/.." && (git rev-parse --is-inside-work-tree | grep -q true)) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --show-toplevel) 2>&1
DEBUG:helpers.hsystem:> (git branch --show-current) 2>&1
DEBUG:helpers.hsystem:> (git rev-parse --short HEAD) 2>&1
DEBUG:helpers.hsystem:> (git log --date=local --oneline --graph --date-order --decorate --pretty=format:'%h %<(8)%aN%  %<(65)%s (%>(14)%ar) %ad %<(10)%d' -3) 2>&1


INFO:__main__:# Git
  branch_name='CmTask11085_Money_20_extract_email'
  hash='4eb17323e'
  # Last commits:
    * 4eb17323e Shayan   integrate GitHub app approach (#11053)                            (    3 days ago) Wed Dec 18 17:11:23 2024  (HEAD -> CmTask11085_Money_20_extract_email, origin/master, origin/HEAD, master)
    * 9220c3158 Nina Lee CmTask10968_C11a.config3_shadow_trading_DAG_reconciliation_fails (#11014) (    4 days ago) Tue Dec 17 22:21:43 2024           
    * 2d9a645b1 Heanh Sok CmampTask10224_Make_infra_dir_releasable_3 (#10972)               (    4 days ago) Tue Dec 17 21:08:14 2024           
# Machine info
  system=Linux
  node name=a4af410f3554
  release=5.15.0-1072-aws
  version=#78~20.04.1-Ubuntu SMP Wed Oct 9 15:30:47 UTC 2024
  machine=x86_64
  processor=x86_64
  cpu count=8
  cpu freq=scpufreq(current=2499.998, min=0.0, max=0.0)
  memory=svmem(total=33280225280, available=25236455424, percent=24.2, used=7555588096, free=3470036992, active=7284064256, inactive

In [6]:
# Google Drive Setup.
google_creds_path = "service.json"
google_sheet_helper = GoogleSheetsHelper(google_creds_path)

In [8]:
file_id = "1--ERHzSPVWgfFqjXQwGhFv4siH-RJIwoNr6iZC1pHJ8"
sheet = google_sheet_helper.google_account.open_by_key(file_id)
money_df = google_sheet_helper.read_sheet(file_id)

INFO:root:Opened spreadsheet with ID: 1--ERHzSPVWgfFqjXQwGhFv4siH-RJIwoNr6iZC1pHJ8
INFO:root:Selected worksheet: Sheet1


In [9]:
len(money_df)

500

In [11]:
money_df.tail()

,fullname,companyName,jobTitle,location,Notes,Skip,Buyer/Software,Insurance,Bank,Lending,Cloud Partner,ISV,VC?,Invest
495,Deanna Blaise,Valley Strong Credit Union,SVP Member Business Services,"Bakersfield, United States",,,,,✔,,,,,
496,Jonas Jacobi,ValidMind,CEO & Co-founder,"Palo Alto, United States",,,,,,,,,,
497,Charles Fischer,ValidMind,Chief Revenue Officer,None,,,,,,,,,,
498,Michael Roenning,ValidMind,"Director, Solution Architecture",None,,,,,,,,,,
499,Greg Rable,ValidiFI,CEO,None,,,,,,,,,,


In [13]:
money_df[["firstName", "lastName"]] = money_df["fullname"].str.split(
    " ", n=1, expand=True
)

In [15]:
title_clean = "cleaned_profiles"
cleaned_profiles_tab = sheet.add_worksheet(
    title=title_clean, rows="100", cols="20"
)
google_sheet_helper.write_results(file_id, money_df, title_clean)

INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: cleaned_profiles


In [16]:
money_df.tail()

,fullname,companyName,jobTitle,location,Notes,Skip,Buyer/Software,Insurance,Bank,Lending,Cloud Partner,ISV,VC?,Invest,firstName,lastName
495,Deanna Blaise,Valley Strong Credit Union,SVP Member Business Services,"Bakersfield, United States",,,,,✔,,,,,,Deanna,Blaise
496,Jonas Jacobi,ValidMind,CEO & Co-founder,"Palo Alto, United States",,,,,,,,,,,Jonas,Jacobi
497,Charles Fischer,ValidMind,Chief Revenue Officer,None,,,,,,,,,,,Charles,Fischer
498,Michael Roenning,ValidMind,"Director, Solution Architecture",None,,,,,,,,,,,Michael,Roenning
499,Greg Rable,ValidiFI,CEO,None,,,,,,,,,,,Greg,Rable


In [19]:
merged_df = cmhuhuap.hunter_drop_emails(
    "firstName",
    "lastName",
    "companyName",
    title_clean,
    hunter_api_key,
    google_creds_path,
    file_id,
    dropcontact_api_key,
)

INFO:ck_marketing.hunterio.hunter_api:Starting process to read records and find emails
INFO:root:Opened spreadsheet with ID: 1--ERHzSPVWgfFqjXQwGhFv4siH-RJIwoNr6iZC1pHJ8
INFO:root:Selected worksheet: cleaned_profiles
INFO:ck_marketing.hunterio.hunter_api:Finding bulk emails using company name
INFO:ck_marketing.hunterio.hunter_api:Writing results to Google Sheets
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: hunter_results
INFO:ck_marketing.hunterio.hunter_api:Total records processed: 500
INFO:ck_marketing.hunterio.hunter_api:Emails found: 295
INFO:ck_marketing.hunterio.hunter_api:Emails not found: 205
INFO:ck_marketing.hunterio.hunter_api:Number of unique companies: 226
INFO:ck_marketing.hunterio.hunter_api:Percentage of emails found: 59.00%
INFO:ck_marketing.hunterio.hunter_api:Companies by number of records:
INFO:ck_marketing.hunterio.hunter_api:companyName
Visa                                                                          

Starting query batch 0.
Batch 0: Query ID: ynldenxysrzzken.


Processing batches:  20%|#########                                    | 1/5 [01:34<06:17, 94.37s/it]

Batch 0: Query finished. Credits left: 5726.
Batch 0 completed in 94.37 seconds.
Starting query batch 1.
Batch 1: Query ID: cmotmtoejefbifq.


Processing batches:  40%|##################                           | 2/5 [02:58<04:24, 88.20s/it]

...

Processing batches:  80%|####################################         | 4/5 [05:25<01:17, 77.23s/it]

Batch 3: Query finished. Credits left: 5576.
Batch 3 completed in 63.29 seconds.
Starting query batch 4.
Batch 4: Query ID: fztmdoguuqclaxd.


Processing batches: 100%|#############################################| 5/5 [05:56<00:00, 71.32s/it]
INFO:ck_marketing.hunterio.hunter_api:Number of emails found : 132
INFO:ck_marketing.hunterio.hunter_api:Number of profile emails hunter could not find 205:


Batch 4: Query finished. Credits left: 5571.
Batch 4 completed in 30.97 seconds.
Total processing time: 356.60 seconds.


INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: hunter_drop_emails
INFO:ck_marketing.hunterio.hunter_api:Email extraction completed. Results saved in the new tab: all_emails
